# 00 DOSM Raw Data To Clean Poverty Labels

This notebook records the raw DOSM poverty-label lineage used by the final Malaysia state-level modelling panel. It keeps the raw DOSM state poverty table separate from the model-ready panel so the repository workflow is clear: raw official labels are cleaned, then validated against the retained panel used by downstream modelling notebooks.

## Purpose

- Retrieve the official DOSM state poverty dataset for Malaysia.
- Save an untouched raw copy under `data/raw/dosm/`.
- Create a cleaned state-year poverty-label table under `data/interim/`.
- Validate the cleaned labels against the target columns in `data/state_year_panel_modelready_2002_2022.csv`.

The satellite and geospatial predictors remain included as derived state-level features in the model-ready panel, with provenance documented separately. This notebook repairs and demonstrates the DOSM poverty-label lineage only.

In [1]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "dosm"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
QA_DIR = PROJECT_ROOT / "outputs" / "qa"
for directory in [RAW_DIR, INTERIM_DIR, QA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DOSM_DATASET_ID = "hh_poverty_state"
DOSM_API_URL = "https://api.data.gov.my/data-catalogue?id=hh_poverty_state&limit=10000"
DOSM_CSV_URL = "https://storage.dosm.gov.my/hies/hh_poverty_state.csv"
RAW_PATH = RAW_DIR / "hh_poverty_state.csv"
CLEAN_PATH = INTERIM_DIR / "dosm_poverty_state_clean.csv"
LINEAGE_QA_PATH = QA_DIR / "dosm_raw_lineage_validation.json"
PANEL_PATH = PROJECT_ROOT / "data" / "state_year_panel_modelready_2002_2022.csv"

SURVEY_YEARS = [2002, 2004, 2007, 2009, 2012, 2014, 2016, 2019, 2020, 2022]
STATE_KEY_MAP = {
    "Johor": "Johor",
    "Kedah": "Kedah",
    "Kelantan": "Kelantan",
    "Melaka": "Melaka",
    "Negeri Sembilan": "NegeriSembilan",
    "Pahang": "Pahang",
    "Perak": "Perak",
    "Perlis": "Perlis",
    "Pulau Pinang": "PulauPinang",
    "Sabah": "Sabah",
    "Sarawak": "Sarawak",
    "Selangor": "Selangor",
    "Terengganu": "Trengganu",
    "W.P. Kuala Lumpur": "KualaLumpur",
    "W.P. Labuan": "Labuan",
    "W.P. Putrajaya": "Putrajaya",
}
TARGET_COLUMNS = ["poverty_absolute", "poverty_hardcore", "poverty_relative"]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## Step 1 - Retrieve And Retain Official DOSM Raw Data

The official OpenDOSM catalogue exposes this dataset as `hh_poverty_state`. The notebook retrieves it through the official `api.data.gov.my` catalogue endpoint and records the storage CSV URL in the metadata for traceability. If the network is unavailable but the raw file already exists locally, the notebook falls back to the retained raw copy.

In [2]:
expected_columns = ["date", "state", "poverty_absolute", "poverty_hardcore", "poverty_relative"]
retrieval_mode = "api.data.gov.my"
try:
    response = requests.get(DOSM_API_URL, timeout=60)
    response.raise_for_status()
    raw_df = pd.DataFrame(response.json())
except Exception as exc:
    if not RAW_PATH.exists():
        raise RuntimeError(
            "Could not retrieve DOSM raw data and no retained raw file exists. "
            "Check the network connection or download hh_poverty_state.csv manually."
        ) from exc
    raw_df = pd.read_csv(RAW_PATH)
    retrieval_mode = "retained_local_copy"

missing_columns = set(expected_columns) - set(raw_df.columns)
if missing_columns:
    raise ValueError(f"DOSM raw data missing expected columns: {sorted(missing_columns)}")

raw_df = raw_df[expected_columns].copy()
raw_df.to_csv(RAW_PATH, index=False)

raw_metadata = {
    "dataset_id": DOSM_DATASET_ID,
    "api_url": DOSM_API_URL,
    "csv_url": DOSM_CSV_URL,
    "retrieval_mode": retrieval_mode,
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "raw_path": str(RAW_PATH.relative_to(PROJECT_ROOT)),
    "rows": int(len(raw_df)),
    "columns": list(raw_df.columns),
    "sha256": sha256_file(RAW_PATH),
}

summary = pd.DataFrame([{
    "raw_path": raw_metadata["raw_path"],
    "rows": raw_metadata["rows"],
    "state_count": raw_df["state"].nunique(),
    "date_min": raw_df["date"].min(),
    "date_max": raw_df["date"].max(),
    "retrieval_mode": retrieval_mode,
}])
display(summary)
raw_df.head()

,raw_path,rows,state_count,date_min,date_max,retrieval_mode
0,data\raw\dosm\hh_poverty_state.csv,294,16,1970-01-01,2022-01-01,api.data.gov.my


,date,state,poverty_absolute,poverty_hardcore,poverty_relative
0,1970-01-01,Johor,45.7,NaN,NaN
1,1976-01-01,Johor,29.0,NaN,NaN
2,1979-01-01,Johor,18.2,NaN,NaN
3,1984-01-01,Johor,12.2,3.1,NaN
4,1987-01-01,Johor,11.1,2.6,NaN


## Step 2 - Clean DOSM Labels For Project Survey Years

This step converts the DOSM `date` column to a survey `year`, maps DOSM state names to the canonical `state_key`, filters to the 10 project survey years, and writes a clean poverty-label table. No model-derived values are created here.

In [3]:
clean = raw_df.copy()
clean["date"] = pd.to_datetime(clean["date"], errors="coerce")
clean["year"] = clean["date"].dt.year
clean["state_raw"] = clean["state"]
clean["state_key"] = clean["state_raw"].map(STATE_KEY_MAP)

if clean["state_key"].isna().any():
    unknown = sorted(clean.loc[clean["state_key"].isna(), "state_raw"].dropna().unique())
    raise ValueError(f"Unmapped DOSM state names: {unknown}")

clean = clean.loc[clean["year"].isin(SURVEY_YEARS), [
    "state_key", "state_raw", "year", *TARGET_COLUMNS
]].sort_values(["state_key", "year"]).reset_index(drop=True)
clean.to_csv(CLEAN_PATH, index=False)

clean_summary = pd.DataFrame([{
    "clean_path": str(CLEAN_PATH.relative_to(PROJECT_ROOT)),
    "rows": int(len(clean)),
    "state_count": clean["state_key"].nunique(),
    "survey_years": ", ".join(str(year) for year in sorted(clean["year"].unique())),
    "sha256": sha256_file(CLEAN_PATH),
}])
display(clean_summary)
clean.head()

,clean_path,rows,state_count,survey_years,sha256
0,data\interim\dosm_poverty_state_clean.csv,158,16,"2002, 2004, 2007, 2009, 2012, 2014, 2016, 2019...",61726709cd69ee4a548342898d2ce6fb77cb469eaf4d71...


,state_key,state_raw,year,poverty_absolute,poverty_hardcore,poverty_relative
0,Johor,Johor,2002,1.8,0.2,16.1
1,Johor,Johor,2004,2.0,0.3,15.3
2,Johor,Johor,2007,1.5,0.2,14.2
3,Johor,Johor,2009,1.3,0.1,17.2
4,Johor,Johor,2012,0.9,0.1,16.1


## Step 3 - Validate Clean Labels Against The Model-Ready Panel

The model-ready panel is the stable downstream input for the existing modelling notebooks. This validation confirms that its poverty target columns match the cleaned DOSM labels for all shared state-year rows. Extra DOSM state-year labels not present in the panel are reported rather than silently discarded.

In [4]:
panel = pd.read_csv(PANEL_PATH)
panel_targets = panel[["state_key", "state", "year", *TARGET_COLUMNS]].copy()
merged = clean.merge(
    panel_targets,
    on=["state_key", "year"],
    how="outer",
    suffixes=("_clean", "_panel"),
    indicator=True,
)

mismatches = []
for target in TARGET_COLUMNS:
    clean_col = f"{target}_clean"
    panel_col = f"{target}_panel"
    both_missing = merged[clean_col].isna() & merged[panel_col].isna()
    both_equal = (merged[clean_col] - merged[panel_col]).abs() <= 1e-9
    bad = merged.loc[~(both_missing | both_equal), ["state_key", "year", clean_col, panel_col, "_merge"]]
    for _, row in bad.iterrows():
        mismatches.append({
            "target": target,
            "state_key": row["state_key"],
            "year": int(row["year"]),
            "clean_value": None if pd.isna(row[clean_col]) else float(row[clean_col]),
            "panel_value": None if pd.isna(row[panel_col]) else float(row[panel_col]),
            "merge_status": row["_merge"],
        })

qa = {
    "lineage": "DOSM raw state poverty data -> cleaned DOSM labels -> retained model-ready panel target columns",
    "raw": raw_metadata,
    "clean": {
        "path": str(CLEAN_PATH.relative_to(PROJECT_ROOT)),
        "rows": int(len(clean)),
        "state_count": int(clean["state_key"].nunique()),
        "survey_years": sorted(int(year) for year in clean["year"].unique()),
        "sha256": sha256_file(CLEAN_PATH),
    },
    "panel": {
        "path": str(PANEL_PATH.relative_to(PROJECT_ROOT)),
        "rows": int(len(panel)),
        "state_count": int(panel["state_key"].nunique()),
        "survey_years": sorted(int(year) for year in panel["year"].unique()),
    },
    "validation": {
        "shared_rows": int((merged["_merge"] == "both").sum()),
        "clean_only_rows": int((merged["_merge"] == "left_only").sum()),
        "panel_only_rows": int((merged["_merge"] == "right_only").sum()),
        "mismatch_count": len(mismatches),
        "mismatches": mismatches[:50],
        "status": "PASS" if not mismatches else "FAIL",
    },
}

LINEAGE_QA_PATH.write_text(json.dumps(qa, indent=2), encoding="utf-8")
display(pd.DataFrame([qa["validation"]]).drop(columns=["mismatches"]))

if mismatches:
    raise AssertionError(f"Clean DOSM labels do not match model-ready panel targets: {len(mismatches)} mismatches")

print(f"Wrote lineage QA to {LINEAGE_QA_PATH.relative_to(PROJECT_ROOT)}")

,shared_rows,clean_only_rows,panel_only_rows,mismatch_count,status
0,158,0,0,0,PASS


Wrote lineage QA to outputs\qa\dosm_raw_lineage_validation.json
